In [1]:
import sys
from scipy.io import mmread
import os
import glob
import pandas as pd
import numpy as np
#from pandas_ods_reader import read_ods
from copy import deepcopy
import pprint
import json
import re
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor
from sklearn import preprocessing
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import pdist
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from sklearn import metrics
from sklearn.cluster import DBSCAN
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from collections import Counter
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
import harmonypy as hm
from matplotlib.cm import ScalarMappable
from datetime import date
import mpld3
import hvplot.pandas
import holoviews as hv
from holoviews import opts
import panel as pn
import bokeh
from bokeh.resources import INLINE
from adjustText import adjust_text
from scipy.stats import mannwhitneyu, false_discovery_control, wilcoxon
import pygwalker as pyg


import dimorph_processing as dp
import cell_comparison as cc
import sex_stats as ss

today = str(date.today())
%matplotlib notebook
%load_ext autoreload
%autoreload 2

# Use Pygwalker to explore SD cell type

### Input full name of cluster, paths to FC data and U test data

In [2]:
cluster_fn = 'Vglut2-6-Otp-Zic5'
run = '311224_run'
cell_class = 'Vglut2'
#delta_data_folder = '/bigdata/isaac/gaba_files/sex_stats/gene_delta_plots/171224_run/data/'
#utest_data_folder = '/bigdata/isaac/gaba_files/sex_stats/gene_delta_plots/171224_run/utest_data/'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
utest_data_folder = '/bigdata/isaac/'+cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/utest_data/'


### Load m log expr data, compute deltas, and combine with q values into single dataframe

In [3]:
test_df = pd.read_json(delta_data_folder + cell_class + '_expr_mlog_df_c_' +cluster_fn +'.json')
u_test_BN_m_df = pd.read_json(utest_data_folder + cell_class + '_U_test_BN_m_c_' + cluster_fn +'.json')
u_test_BN_f_df = pd.read_json(utest_data_folder + cell_class + '_U_test_BN_f_c_' + cluster_fn +'.json')
u_test_mf_B_df = pd.read_json(utest_data_folder + cell_class + '_U_test_mf_B_c_' + cluster_fn +'.json')
u_test_mf_N_df = pd.read_json(utest_data_folder + cell_class + '_U_test_mf_N_c_' + cluster_fn +'.json')
delta_B_N_m = test_df['B_m'] - test_df['N_m']
delta_B_N_f = test_df['B_f'] - test_df['N_f']
delta_m_f_N = test_df['N_m'] - test_df['N_f']
delta_m_f_B = test_df['B_m'] - test_df['B_f']
delta_df = pd.DataFrame(index=test_df.index, columns=['delta_B_N_m',
                                                      'delta_B_N_f',
                                                      'delta_m_f_N',
                                                      'delta_m_f_B',
                                                      'abs_delta_B_N_m',
                                                      'abs_delta_B_N_f',
                                                      'abs_delta_m_f_N',
                                                      'abs_delta_m_f_B',
                                                      'q_BN_m',
                                                      'q_BN_f',
                                                      'q_mf_B',
                                                      'q_mf_N',
                                                      '-log10_q_BN_m',
                                                      '-log10_q_BN_f',
                                                      '-log10_q_mf_B',
                                                      '-log10_q_mf_N'])
delta_df.loc[:,'delta_B_N_m'] = delta_B_N_m
delta_df.loc[:,'delta_B_N_f'] = delta_B_N_f
delta_df.loc[:,'delta_m_f_B'] = delta_m_f_B
delta_df.loc[:,'delta_m_f_N'] = delta_m_f_N
delta_df.loc[:,'abs_delta_B_N_m'] = np.abs(delta_B_N_m)
delta_df.loc[:,'abs_delta_B_N_f'] = np.abs(delta_B_N_f)
delta_df.loc[:,'abs_delta_m_f_B'] = np.abs(delta_m_f_B)
delta_df.loc[:,'abs_delta_m_f_N'] = np.abs(delta_m_f_N)
delta_df.loc[:,'q_BN_m'] = np.array(u_test_BN_m_df.loc[:,'p_adj'])
delta_df.loc[:,'q_BN_f'] = np.array(u_test_BN_f_df.loc[:,'p_adj'])
delta_df.loc[:,'q_mf_B'] = np.array(u_test_mf_B_df.loc[:,'p_adj'])
delta_df.loc[:,'q_mf_N'] = np.array(u_test_mf_N_df.loc[:,'p_adj'])
delta_df.loc[:,'-log10_q_BN_m']  = -np.log10(np.array(u_test_BN_m_df.loc[:,'p_adj'])).astype('float64')
delta_df.loc[:,'-log10_q_BN_f']  = -np.log10(np.array(u_test_BN_f_df.loc[:,'p_adj'])).astype('float64')
delta_df.loc[:,'-log10_q_mf_B']  = -np.log10(np.array(u_test_mf_B_df.loc[:,'p_adj'])).astype('float64')
delta_df.loc[:,'-log10_q_mf_N']  = -np.log10(np.array(u_test_mf_N_df.loc[:,'p_adj'])).astype('float64')
delta_df_pyg = delta_df.reset_index()

/tmp/ipykernel_3131757/1041936898.py:40: RuntimeWarning: divide by zero encountered in log10
  delta_df.loc[:,'-log10_q_mf_B']  = -np.log10(np.array(u_test_mf_B_df.loc[:,'p_adj'])).astype('float64')
/tmp/ipykernel_3131757/1041936898.py:41: RuntimeWarning: divide by zero encountered in log10
  delta_df.loc[:,'-log10_q_mf_N']  = -np.log10(np.array(u_test_mf_N_df.loc[:,'p_adj'])).astype('float64')


In [4]:
delta_df_pyg

,index,delta_B_N_m,delta_B_N_f,delta_m_f_N,delta_m_f_B,abs_delta_B_N_m,abs_delta_B_N_f,abs_delta_m_f_N,abs_delta_m_f_B,q_BN_m,q_BN_f,q_mf_B,q_mf_N,-log10_q_BN_m,-log10_q_BN_f,-log10_q_mf_B,-log10_q_mf_N
0,0610007P14Rik,-0.181494,-0.263609,-0.065289,0.016826,0.181494,0.263609,0.065289,0.016826,0.167698,0.065576,0.941177,0.843285,0.775471,1.183255,0.026329,0.074026
1,0610009B22Rik,-0.250643,-0.150757,0.235995,0.136109,0.250643,0.150757,0.235995,0.136109,0.020809,0.108082,0.303901,0.24282,1.681753,0.966248,0.517268,0.614715
2,0610009L18Rik,-0.028096,-0.094243,-0.002856,0.063291,0.028096,0.094243,0.002856,0.063291,0.804582,0.239888,0.601572,0.974503,0.09443,0.619991,0.220712,0.011217
3,0610009O20Rik,-0.03967,0.079925,0.111892,-0.007703,0.03967,0.079925,0.111892,0.007703,0.861122,0.374341,0.9846,0.516075,0.064935,0.426733,0.00674,0.287287
4,0610010F05Rik,-0.167145,-0.115664,0.105948,0.054467,0.167145,0.115664,0.105948,0.054467,0.128468,0.349566,0.837481,0.692606,0.891206,0.456471,0.077025,0.159514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7240,Zyg11b,-0.258106,-0.139556,0.108134,-0.010416,0.258106,0.139556,0.108134,0.010416,0.076556,0.511493,0.997511,0.743205,1.11602,0.29116,0.001082,0.128891
7241,Zzef1,-0.13297,0.003548,0.104616,-0.031902,0.13297,0.003548,0.104616,0.031902,0.336561,0.983353,0.948007,0.719249,0.472937,0.007291,0.023189,0.143121
7242,Zzz3,-0.289621,0.036238,0.137221,-0.188639,0.289621,0.036238,0.137221,0.188639,0.007594,0.828508,0.309343,0.545657,2.119549,0.081703,0.50956,0.26308
7243,l7Rn6,-0.21035,-0.277946,-0.001506,0.06609,0.21035,0.277946,0.001506,0.06609,0.055746,0.020298,0.815687,0.983542,1.253786,1.692541,0.088477,0.007207


In [5]:
walker = pyg.walk(delta_df_pyg, hideDataSourceConfig = True, vegaTheme = 'vega')

Box(children=(HTML(value='\n<div id="ifr-pyg-00062f1b675f2e9ayNVYQfpb8gn19Szv" style="height: auto">\n    <hea…

In [131]:
np.array(-np.log10(delta_df.loc[:,'q_mf_N'].astype('float64')))

array([-0., -0., -0., ..., -0., -0., -0.])

In [130]:
-np.log10(np.array(u_test_mf_N_df.loc[:,'p_adj'])).astype('float64')

array([-0., -0., -0., ..., -0., -0., -0.])

In [157]:
np.any(np.array(-np.log10(delta_df.loc[:,'q_mf_N'].astype('float64'))) == -np.log10(np.array(u_test_mf_N_df.loc[:,'p_adj'])).astype('float64'))

/home/isaac/anaconda3/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_3560256/4159045840.py:1: RuntimeWarning: divide by zero encountered in log10
  np.any(np.array(-np.log10(delta_df.loc[:,'q_mf_N'].astype('float64'))) == -np.log10(np.array(u_test_mf_N_df.loc[:,'p_adj'])).astype('float64'))


True

In [162]:
-np.log10(delta_df.loc[:,'q_mf_N'].astype('float64'))

/home/isaac/anaconda3/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


0610007P14Rik    0.000680
0610009B22Rik    0.000680
0610009L18Rik    0.000680
0610009O20Rik    0.000680
0610010F05Rik    0.000680
                   ...   
Zyg11b           0.160058
Zzef1            0.388535
Zzz3             0.000680
l7Rn6            0.000680
mt-Nd6           0.000680
Name: q_mf_N, Length: 6701, dtype: float64

In [163]:
#in case where no sig genes for both delta mf _N and _B, use volcano df for single delta isolation/visualizion
volcano_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/volcano_plots/data/'
test = 'mf_N'
v_df = pd.read_json(volcano_data_folder + cell_class + '_v_'+ test +'_df_c_' +cluster_fn +'.json')
v_df_pyg = v_df.reset_index()


In [164]:
v_df_pyg

,index,delta,-log10(p_adj),p_adj
0,0610007P14Rik,-0.065100,0.000680,0.998436
1,0610009B22Rik,0.010293,0.000680,0.998436
2,0610009L18Rik,-0.030181,0.000680,0.998436
3,0610009O20Rik,0.010212,0.000680,0.998436
4,0610010F05Rik,0.076218,0.000680,0.998436
...,...,...,...,...
6696,Zyg11b,0.188725,0.160058,0.691738
6697,Zzef1,0.228832,0.388535,0.408757
6698,Zzz3,0.039004,0.000680,0.998436
6699,l7Rn6,0.022232,0.000680,0.998436


In [165]:
walker_alt = pyg.walk(v_df_pyg, hideDataSourceConfig = True, vegaTheme = 'vega')

Box(children=(HTML(value='\n<div id="ifr-pyg-00062a3e9ade453bCYge0hxJT9ZArHfc" style="height: auto">\n    <hea…

In [7]:
test_df

,N_f,B_f,N_m,B_m
0610007P14Rik,0.723856,0.460247,0.658568,0.477073
0610009B22Rik,0.303184,0.152427,0.539179,0.288537
0610009L18Rik,0.158529,0.064286,0.155673,0.127577
0610009O20Rik,0.078431,0.158357,0.190324,0.150654
0610010F05Rik,0.340734,0.225070,0.446682,0.279537
...,...,...,...,...
Zyg11b,0.853328,0.713772,0.961462,0.703356
Zzef1,0.462466,0.466014,0.567082,0.434112
Zzz3,0.445437,0.481676,0.582658,0.293036
l7Rn6,0.578623,0.300677,0.577117,0.366767


In [15]:
dp_markers = ['Esr1','Esr2','Ar','Pgr']
other_markers = ['Uba52']
dp_markers_in_test_df = []
for g in dp_markers:
    if g in test_df.index:
        dp_markers_in_test_df.append(g)

In [19]:
dp_markers_in_test_df

['Esr1', 'Ar', 'Pgr']

In [17]:
np.where(test_df.index == 'Esr1')

(array([2075]),)

In [22]:
df = test_df.loc[dp_markers_in_test_df]

In [20]:
'Uba52' in test_df.index

True

In [24]:
df.plot(kind='bar', width=0.8, figsize=(8, 4), colormap='Set2')
plt.ylabel('Expression')
plt.title('Vglut2-6-Otp-Zic5 - Mean Log Expression by group')
plt.xticks(rotation=0)
plt.legend(title='Group')
plt.tight_layout()
plt.savefig('/bigdata/isaac/Vglut2_files/sex_stats/tmp/' + 'dp_markers.png')
plt.show()

<IPython.core.display.Javascript object>

In [9]:
# Data as a dictionary
data = {
    'Gene': ['Esr1', 'Esr1', 'Esr1', 'Esr1', 'Ar', 'Ar', 'Ar', 'Ar', 'Pgr', 'Pgr', 'Pgr', 'Pgr'],
    'Condition': ['N', 'B', 'N', 'B', 'N', 'B', 'N', 'B', 'N', 'B', 'N', 'B'],
    'Sex': ['f', 'f', 'm', 'm', 'f', 'f', 'm', 'm', 'f', 'f', 'm', 'm'],
    'Expression': [0.419919, 0.518487, 0.333475, 0.216692, 0.672434, 0.649127, 1.119253, 0.764769, 
                   0.156863, 0.180999, 0.390487, 0.235269]
}

# Create DataFrame
df = pd.DataFrame(data)

# Set up the plot
sns.set(style="whitegrid")
g = sns.relplot(data=df, x="Condition", y="Expression", hue="Sex", col="Gene", kind="line", 
                marker="o", height=3, aspect=1, palette="Set2", linewidth=2)

# Adjust layout and show
plt.tight_layout()
plt.savefig('/bigdata/isaac/Vglut2_files/sex_stats/tmp/' + 'dp_markers_snsrel.png')
plt.show()

<IPython.core.display.Javascript object>

In [25]:
raw_df = pd.read_json(delta_data_folder + cell_class + '_expr_raw_df_c_' +cluster_fn +'.json')

In [26]:
raw_df

,AAGGAATGTCTTAGTG-1_10X35_1,GCACTAAAGAACCGCA-1_10X35_1,AGGAATAAGATTCGCT-1_10X35_2,TTCCACGAGAGTTGAT-1_10X35_1,CTTCCGAGTGCAATAA-1_10X35_1,CATGGTACAGGCCTGT-1_10X35_1,AATGGAATCAGCCCAG-1_10X35_1,TTAGTCTGTTCGGTAT-1_10X38_2,TGAACGTAGAGCGACT-1_10X38_2,GACTCAACAATGGGTG-1_10X38_2,...,GTGAGTTTCGATGCTA-1_10X52_3,GTAGGAGTCTCTAAGG-1_10X52_3,CCTCCAATCGGTAGGA-1_10X52_3,CCTCCAATCGAAGCAG-1_10X52_3,GACTGATCACCACTGG-1_10X52_3,CATTTCACACCAATTG-1_10X51_3,GCACTAAAGAAGGTAG-1_10X52_4,AATGAAGCATGACAGG-1_10X52_3,ATCACAGTCCGATGTA-1_10X52_3,CCCTCAACATCGTTCC-1_10X52_4
0610007P14Rik,1,4,1,0,1,0,1,2,0,1,...,0,2,1,0,0,0,1,1,0,0
0610009B22Rik,0,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
0610009L18Rik,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
0610009O20Rik,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
0610010F05Rik,0,0,0,0,0,1,0,0,2,0,...,0,0,1,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zyg11b,2,0,1,1,0,1,3,0,1,1,...,1,1,1,3,0,0,0,1,2,1
Zzef1,1,0,0,1,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
Zzz3,0,0,0,0,1,0,0,0,1,0,...,0,2,0,0,0,0,1,1,0,1
l7Rn6,0,1,2,0,1,0,0,3,0,0,...,0,0,0,0,1,1,0,0,0,0
